In [53]:
import gymnasium as gym
import numpy as np
from tetris_gymnasium.envs.tetris import Tetris
from tetris_gymnasium.wrappers.observation import RgbObservation, FeatureVectorObservation
from tetris_gymnasium.wrappers.grouped import GroupedActionsObservations
from gymnasium.wrappers import TimeLimit, ResizeObservation, RecordVideo, FrameStackObservation, GrayscaleObservation
from stable_baselines3 import DQN, PPO
import os

In [54]:
RENDER_ENV = False
LOAD_MODEL = False
new_size = (96,136) #(84,84)
batch_size = 32
num_episodes = 100 #43200
max_episode_steps = 100
num_stacked_frames = 4
intervals = 4
Model = "PPO" # DQN o PPO
version = 1

In [55]:
def get_last_modified_file(directory_path):
    if not os.path.isdir(directory_path):
        print(f"Error: Directory '{directory_path}' does not exist.")
        return None
    files = [os.path.join(directory_path, f) for f in os.listdir(directory_path) if os.path.isfile(os.path.join(directory_path, f))]
    if not files:
        return None
    files.sort(key=os.path.getmtime, reverse=True)
    return files[0]

target_directory = f"../Models_Saves/{Model}"  # Replace with your directory path
model_load_path = get_last_modified_file(target_directory)

if model_load_path:
    print(f"The last modified file is: {model_load_path}")
else:
    print("No files found in the directory or directory does not exist.")

The last modified file is: ../Models_Saves/PPO/PPO_V1_S10000.zip


In [56]:
try:
    os.mkdir("../Models_Saves")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Models_Saves/PPO")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Models_Saves/DQN")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Video_Tetris_IA")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Video_Tetris_IA/PPO")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Video_Tetris_IA/DQN")
except Exception as e:
    print(f"Error: {e}")

Error: [Errno 17] File exists: '../Models_Saves'
Error: [Errno 17] File exists: '../Models_Saves/PPO'
Error: [Errno 17] File exists: '../Models_Saves/DQN'
Error: [Errno 17] File exists: '../Video_Tetris_IA'
Error: [Errno 17] File exists: '../Video_Tetris_IA/PPO'
Error: [Errno 17] File exists: '../Video_Tetris_IA/DQN'


In [57]:
def calc_max_height(mat, height=1):
  mat1 = np.rot90(mat)
  mat1 = np.rot90(mat1)
  act_height=0
  not_seen_height=0
  for row in mat:
    act_height+=1
    if act_height < height-4:
      continue
    if act_height > height+4:
      return(height)
    for col in row:
      if col > 0:
        return(act_height)
  return height

def calc_holes(mat,height):
  mat = np.rot90(mat)
  holes = 0
  for row in mat:
    countdown = 20-height
    flag = False
    for col in row:
      countdown-=1
      if countdown < 0:
        if col > 0:
          flag = True
        elif flag:
          holes+=1
  return holes

In [58]:
try:
  env.close()
except:
  print('no hay env para cerrar')

In [59]:

class CustomRewardWrapper(gym.RewardWrapper):
    """
    Custom reward shaping to encourage forward movement.
    This wrapper modifies the reward based on the agent's horizontal position.
    """
    def __init__(self, env, holes_penalty=-0.005, heigh_penalty=-0.2):
        super(CustomRewardWrapper, self).__init__(env)
        self.holes_penalty = holes_penalty
        self.heigh_penalty = heigh_penalty

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)

        game_variables = self.env.unwrapped.get_state().board[:-4, 4:-4]
        self.previous_max_height = calc_max_height(game_variables, 1)

        return obs, info

    def reward(self, reward):
        #print(f"Reward original: {reward}")
        # Probar mayor penalizacion de agujeros
        custom_reward = reward
        game_variables = self.env.unwrapped.get_state().board[:-4, 4:-4]

        if game_variables.any():
            current_max_height = calc_max_height(game_variables, self.previous_max_height)
            current_holes = calc_holes(game_variables, current_max_height)
            if current_max_height > self.previous_max_height:
              custom_reward+=self.heigh_penalty
            custom_reward += current_holes*self.holes_penalty
            self.previous_max_height = current_max_height
        return custom_reward

In [60]:
def make_env(*, game, max_episode_steps=4500, **kwargs):
    env = gym.make(game, **kwargs)
    env = RgbObservation(env)
    #env = FeatureVectorObservation(env)
    env = ResizeObservation(env, new_size)
    env = GrayscaleObservation(env)
    env = FrameStackObservation(env, stack_size=num_stacked_frames)
    env = CustomRewardWrapper(env)
    env.reset(seed=42)
    return env

In [61]:


if __name__ == "__main__":
    env = make_env(game = "tetris_gymnasium/Tetris", render_mode="rgb_array")
    if Model == "DQN":
        model = DQN("CnnPolicy", env, verbose=1, buffer_size=10000) # para Mlp usar FeatureVectorObservation para Cnn usar RgbObservation
    else:
        model = PPO("CnnPolicy", env, verbose=1) # para Mlp usar FeatureVectorObservation para Cnn
    if LOAD_MODEL:
        model.load(model_load_path)
    model.learn(total_timesteps=num_episodes*max_episode_steps, log_interval=4)
    model.save(f"../Models_Saves/{Model}/{Model}_V{version}_S{max_episode_steps*num_episodes}")
    env.close()

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 41.6        |
|    ep_rew_mean          | 9.5         |
| time/                   |             |
|    fps                  | 153         |
|    iterations           | 4           |
|    time_elapsed         | 53          |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.018581964 |
|    clip_fraction        | 0.243       |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.02       |
|    explained_variance   | 0.812       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.0146     |
|    n_updates            | 30          |
|    policy_gradient_loss | -0.0373     |
|    value_loss           | 0.158       |
-----------------------------------------


In [62]:
try:
  env = RecordVideo(
    env,
    video_folder=f'../Video_Tetris_IA/{Model}',    # Folder to save videos
    name_prefix=f'{Model}_eval-Trained_steps_{num_episodes*max_episode_steps}',               # Prefix for video filenames
    episode_trigger=lambda x: True    # Record every episode
  )
except Exception as e:
  print(f'error implementando grabacion: {e}')

In [63]:
for episode in range(10):
  state, info = env.reset()
  total_reward = 0
  done = False
  step_count = 0
  while not done:
    step_count+=1
    action, _states = model.predict(state, deterministic=True)
    state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    total_reward += reward
  print(f"Episode: {episode} Reward: {total_reward} Steps: {step_count}")

Episode: 0 Reward: 10.78 Steps: 12
Episode: 1 Reward: 9.76 Steps: 27
Episode: 2 Reward: 10.774999999999999 Steps: 34
Episode: 3 Reward: 8.77 Steps: 15
Episode: 4 Reward: 12.694999999999995 Steps: 46
Episode: 5 Reward: 15.29000000000001 Steps: 98
Episode: 6 Reward: 15.680000000000001 Steps: 107
Episode: 7 Reward: 13.64 Steps: 83
Episode: 8 Reward: 8.795 Steps: 11
Episode: 9 Reward: 9.785 Steps: 11


In [64]:
# var = len(env.unwrapped.get_state().board)
# count = var
# temp = env.unwrapped.get_state().board[:-4, 4:-4]
# print(temp)
# print(calc_max_height(temp))
# print(calc_holes(temp))
# #print(len(env.unwrapped.get_state().board))

In [65]:
# from stable_baselines3.common.env_checker import check_env
# check_env(env)